In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import pickle
import re
from itertools import combinations
from functools import reduce
from typing import Tuple, Any
import seaborn as sns
from scipy.stats import spearmanr, kendalltau
from scipy.spatial.distance import jensenshannon
from scipy.cluster.hierarchy import linkage, leaves_list
from sklearn.metrics import roc_auc_score, ndcg_score, r2_score

### Scoring functions of interest:

* Spearman correlation
* Kendall correlation? for ranking mutations
* Log likelihood
* Log odds against WT
* ROC-AUC
* Normalized Discounted Cumulative Gain (NDCG) for ranking mutations

In [ ]:
groups = pd.read_csv('../DMS_substitutions_groups.tsv',sep='\t')
g1_df = groups[groups['group1_member']==1].rename(columns={'group1_str':'group_str'})
g2_df = groups[groups['group2_member']==1].rename(columns={'group2_str':'group_str'})
csv_dir = '/home/pwoolley/work/proteingym/DMS_ProteinGym_substitutions'

### Functions and Classes for model predictions

In [ ]:
class ProteinMutationModel:
    AMINO_ACIDS = ['A','R','N','D','C','Q','E','G','H','I','L','K','M','F','P','S','T','W','Y','V']

    def __init__(self, temperature=1.0, eps=1e-12):
        self.temperature = temperature
        self.eps = eps
        self.probs_long = None
        self.entropy_df = None
        self.data = None

    # ------------------------------------------------------------------
    # Static utilities
    # ------------------------------------------------------------------

    @staticmethod
    def softmax(x, axis=-1, temperature=1.0):
        if temperature <= 0:
            raise ValueError("Temperature must be positive.")
        x_scaled = x / temperature
        e_x = np.exp(x_scaled - np.max(x_scaled, axis=axis, keepdims=True))
        return e_x / np.sum(e_x, axis=axis, keepdims=True)

    @staticmethod
    def entropy(prob_dist):
        prob_dist = np.asarray(prob_dist)
        prob_dist = prob_dist[prob_dist > 0]
        return -np.sum(prob_dist * np.log2(prob_dist))

    @staticmethod
    def parse_mutations(
        mutation_string: Any
    ) -> Tuple[Any, int, str, str]:

        if pd.isna(mutation_string) or not isinstance(mutation_string, str):
            return [], 0, "", ""

        muts = mutation_string.split(':')
        pattern = re.compile(r'([A-Za-z])(\d+)([A-Za-z])')

        indices, wt, mut = [], "", ""

        for m in muts:
            match = pattern.search(m)
            if match:
                wt += match.group(1)
                mut += match.group(3)
                indices.append(int(match.group(2)))

        if len(indices) == 1:
            return indices[0], 1, wt, mut

        return indices, len(indices), wt, mut

    # ------------------------------------------------------------------
    # Model output handling
    # ------------------------------------------------------------------

    def load_pickle(self, pkl_path: str, protein_id: str):
        with open(pkl_path, "rb") as f:
            pkldata = pickle.load(f)
        for k, v in pkldata.items():
            if protein_id in k:
                indices = [int(x) for x in k.split('indices_')[-1].split('_')]
                probs = self.softmax(v, temperature=self.temperature)
                df = pd.DataFrame(probs,index=indices,columns=self.AMINO_ACIDS)
                self.probs_long = (df.stack().reset_index()
                                   .rename(columns={"level_0": "index","level_1": "amino_acid",0: "prob"}))
                return
        raise ValueError(f"Protein ID '{protein_id}' not found in pickle.")

    def compute_entropy(self):
        self.entropy_df = (
            self.probs_long
            .groupby("index")["prob"]
            .apply(self.entropy)
            .reset_index(name="entropy")
        )

    # ------------------------------------------------------------------
    # Experimental data integration
    # ------------------------------------------------------------------

    def load_experiment(self, csv_path: str):
        exp = pd.read_csv(csv_path)
        exp[['index','number_mut','wt','mut']] = (exp['mutant'].apply(lambda x: pd.Series(self.parse_mutations(x))))
        self.exp_df = exp[exp['number_mut'] == 1].reset_index(drop=True)

    def build_dataset(self):
        probs = self.probs_long.copy()
        exp_df = self.exp_df.copy()
        # Rank probabilities
        probs['prob'] = probs['prob'].astype('float64')
        probs = probs.sort_values(['index','prob'], ascending=[True, False])
        probs['rank'] = probs.groupby('index')['prob'].rank(method='first', ascending=False)
        # WT info
        wt = exp_df[['index','wt']].drop_duplicates()
        wt_prob = pd.merge(wt, probs,left_on=['index','wt'],right_on=['index','amino_acid']).rename(columns={'prob':'wt_prob'})[['index','wt_prob']]
        wt_rank = pd.merge(wt, probs,left_on=['index','wt'],right_on=['index','amino_acid']).rename(columns={'rank':'wt_rank'})[['index','wt_rank']]
        # Mutation info
        exp = pd.merge(exp_df, probs,left_on=['index','mut'],right_on=['index','amino_acid']).rename(columns={'prob':'mut_prob','rank':'mut_rank'})
        # Merge everything
        exp = (
            exp
            .merge(wt_prob, on='index')
            .merge(wt_rank, on='index')
            .merge(self.entropy_df, on='index')
        )
        exp = exp.rename(columns={'DMS_score':'exp'})
        exp['log_odds'] = (np.log(exp['mut_prob'].clip(self.eps))-np.log(exp['wt_prob'].clip(self.eps)))
        self.data = exp.reset_index(drop=True)
        

    # ------------------------------------------------------------------
    # Evaluation metrics
    # ------------------------------------------------------------------

    def spearman(self, column='exp', subsetting = None, score_col='log_odds'):
        """
        Subset the data where `column` is between min_val and max_val, then compute Spearman correlation
        with `score_col`.
        Parameters
        ----------
        column : str
            Column name to filter on (default 'exp').
        subsetting : dict
            Keys are variables in the dataframe, values are a dict with two keys:
                min_val : float
                    Minimum value (inclusive) for filtering. If None, no lower bound.
                max_val : float
                    Maximum value (inclusive) for filtering. If None, no upper bound.
        score_col : str
            Column name to compute Spearman correlation against (default 'log_odds').
        Returns
        -------
        float
            Spearman correlation of the filtered data. Returns np.nan if not enough data points.
        """
        if self.data is None:
            raise ValueError("Data not built yet. Run `build_dataset()` first.")
        df = self.data.copy()
        if subsetting is not None:
            for k,v in subsetting.items():
                if v['min_val'] is not None:
                    df = df[df[k] >= v['min_val']]
                if v['max_val'] is not None:
                    df = df[df[k] <= v['max_val']]
        if df.shape[0] < 2:
            return np.nan  # Not enough points to compute correlation
        corr = spearmanr(df[column], df[score_col]).correlation
        return corr
    
    def metrics(self, roc_threshold=0.0, ndcg_k=None):
        df = self.data.copy()
        df['log_likelihood'] = np.log(df['mut_prob'].clip(self.eps))
        out = {
            "spearman": spearmanr(df['exp'], df['log_odds']).correlation,
            "kendall": kendalltau(df['exp'], df['log_odds']).correlation,
            "loglik_spearman": spearmanr(df['exp'], df['log_likelihood']).correlation
        }
        labels = (df['exp'] > roc_threshold).astype(int)
        if labels.nunique() > 1:
            out["roc_auc"] = roc_auc_score(labels, df['log_odds'])
        else:
            out["roc_auc"] = np.nan
        y_true = df['exp'].to_numpy().reshape(1, -1)
        y_score = df['log_odds'].to_numpy().reshape(1, -1)
        minval = y_true.min()
        if minval <= 0:
            y_true = (y_true - minval) + self.eps
        out["ndcg"] = ndcg_score(y_true, y_score, k=ndcg_k)
        return out


### Model prediction file info

In [ ]:
models = {
        #  'esmc_300m.logits.x.combined.pkl':'ESMC_300M',
         'esmc_600m.logits.x.combined.pkl':'ESMC_600M',
        #  'AMPLIFY_120M.logits.x.combined.pkl':'AMPLIFY_120M',
         'AMPLIFY_350M.logits.x.combined.pkl':'AMPLIFY_350M',
        #  'ismc_300m.logits.x.combined.pkl':'ISMC_300M',
         'ismc_600m.logits.x.combined.pkl':'ISMC_600M',
         'SaProt_650M_PDB.logits.x.combined.pkl':'SaProt_650M',
         'esm2_t48_15b_UR50D.logits.x.combined.pkl':'ESM2_15B',
        #  'ESM3_sm_open_v0.structure.logits.x.combined.pkl':'ESM3_sm_structure',
         'ESM3_sm_open_v0.both.logits.x.combined.pkl':'ESM3_sm_both',
        #  'ESM3_sm_open_v0.sequence.logits.x.combined.pkl':'ESM3_sm_sequence',
        #  'ism_t33_650M_uc30pdb.logits.x.combined.pkl':'ISM_650M',
        #  'esm2_t33_650M_UR50D.logits.x.combined.pkl':'ESM2_650M',
         'protein_mpnn.proteinmpnn_v_48_020.logits.x.combined.use_sequence0.pkl':'ProteinMPNN',
        #  'protein_mpnn.proteinmpnn_v_48_020.logits.x.combined.use_sequence1.pkl':'ProteinMPNN_seq1',
         'soluble_mpnn.solublempnn_v_48_020.logits.x.combined.use_sequence0.pkl':'SolubleMPNN',
        #  'soluble_mpnn.solublempnn_v_48_020.logits.x.combined.use_sequence1.pkl':'SolubleMPNN_seq1',
         'ProstT5.logits.x.combined.pkl':'ProstT5'
         }
mapping = {
    "ESM2_650M": "ESM-2 650M",
    "ESM2_3B": "ESM-2 3B",
    "ESMC_600M": "ESMC 600M",
    "ESMC_300M": "ESMC 300M",
    "ESM3_sm_both": "ESM-3 (hybrid)",
    "ESM3_sm_sequence": "ESM-3 (sequence)",
    "ESM3_sm_structure": "ESM-3 (structure)",
    "ISMC_600M": "ISMC 600M",
    "ISMC_300M": "ISMC 300M",
    "ISM_650M": "ISM 650M",
    "ISM2-650M": "ISM 650M",
    "SaProt_650M": "SaProt 650M",
    "AMPLIFY_350M": "AMPLIFY 350M",
    "AMPLIFY_120M": "AMPLIFY 120M",
    "SaProt_650M_AF2": "SaProt 650M (AF2)",
    "ProstT5": "ProstT5",
    "SolubleMPNN": "SolubleMPNN",
    "ProteinMPNN": "ProteinMPNN"
}

pkls_dir = '/home/pwoolley/work/proteingym/outputs/logits/proteingym'
pkls = [os.path.join(pkls_dir,x) for x in os.listdir(pkls_dir) if x.endswith('.pkl')]
csv_dir = '/home/pwoolley/work/proteingym/DMS_ProteinGym_substitutions'


### Figure 2: Dataset analysis (DONE)

In [ ]:
def dataset_analysis_plots(g_df,csv_dir,png_dir,group=1):
    def linear_regression(x,y):
        m, b = np.polyfit(x, y, 1) # Pearson (linear fit)
        y_pred = m * x + b
        pearson_r2 = r2_score(y, y_pred)
        spearman_rho, _ = spearmanr(x, y) # Spearman
        spearman_r2 = spearman_rho**2
        return pearson_r2, spearman_rho, spearman_r2, y_pred
    
    global_stats = []  # store per-protein mean correlations
    if group==1:
        color = '#1f77b4'
    else:
        color = '#ff7f0e'
    for _, row in g_df.iterrows():
        exps = row['group_str'].split(',')
        protein_data = []
        selections = []
        for exp in exps:
            if group==1:
                selection = exp.split(':')[1]
            elif group==2:
                s = exp.split(':')
                selection = f"{s[0].split('_')[2]}_{s[1]}"
            else:
                print('Need to specify group 1 or 2!')
                return
            selections.append(selection)
            csv = os.path.join(csv_dir, exp.split(':')[0])
            exp_data = pd.read_csv(csv)[['mutant', 'DMS_score']]
            exp_data = exp_data.rename(columns={'DMS_score': selection})
            protein_data.append(exp_data)
        protein_data = reduce(lambda left, right: pd.merge(left, right, on='mutant', how='inner'),protein_data)
        pair_stats = [] # --- pairwise correlations ---
        make_plots = True
        for a, b in combinations(selections, 2):
            if b == "Abundance":
                y = protein_data[a]
                x = protein_data[b]
            else:
                x = protein_data[a]
                y = protein_data[b]
            try:
                pearson_r2, spearman_rho, spearman_r2, y_pred = linear_regression(x,y)
                pair_stats.append({'a': a, 'b': b,
                               'pearson': pearson_r2,'spearman_rho': spearman_rho, 'spearman_r2': spearman_r2,
                               'y_pred': y_pred})
            except Exception as e:
                print(e)
                print(x)
                make_plots=False
                break
        if make_plots==False:
            continue
        pair_df = pd.DataFrame(pair_stats)
        worst = pair_df.loc[pair_df['pearson'].idxmin()] # weakest Pearson pair
        a, b, y_pred = worst['a'], worst['b'], worst['y_pred']
        if b == "Abundance":
            pltx,plty = b,a
        else:
            pltx,plty = a,b
        plt.figure(figsize=(4.5, 4))
        ax = plt.gca()
        ax.scatter(protein_data[pltx], protein_data[plty], s=2, c=color)
        if group == 2:
            pltx = f"{pltx.split('_')[0]} {pltx.split('_')[1]}"
            plty = f"{plty.split('_')[0]} {plty.split('_')[1]}"
        ax.set_title(f"{row['name']}", fontsize=16)
        ax.set_xlabel(pltx, fontsize=14)
        ax.set_ylabel(plty, fontsize=14)
        # Limit number of ticks with nice spacing
        ax.xaxis.set_major_locator(ticker.MaxNLocator(nbins=5))
        ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=5))
        ax.tick_params(axis='both', labelsize=14)
        # plt.plot(protein_data[pltx], y_pred, linewidth=1,c='r') # uncomment for trendline
        ax.text(0.05, 0.95,f"Spearman $\\rho$ = {spearman_rho:.3f}", #$\nPearson $R^2 = {pearson_r2:.3f}$
                 transform=plt.gca().transAxes,verticalalignment="top", fontsize=12)
        plt.savefig(os.path.join(png_dir,row['UniProt_ID']+'_spearmanrho.png'),dpi=300,bbox_inches="tight")
        # plt.show()

        # --- store global stats ---# --- store global stats ---
        global_stats.append({
            'name': row['name'],
            'UniProt_ID': row['UniProt_ID'],
            'mean_pearson': pair_df['pearson'].mean(),
            'mean_spearman': pair_df['spearman_rho'].mean(),
            'all_pearson':pair_df['pearson'],
            'all_spearman':pair_df['spearman_rho'],
        })

    # === BAR CHART FOR ALL PROTEINS ===
    stats_df = pd.DataFrame(global_stats).sort_values(['mean_spearman'])
    y = np.arange(len(stats_df))
    plt.figure(figsize=(6, 4))
    plt.barh(y, stats_df['mean_spearman'], label='Spearman $\\rho$',color=color)
    plt.yticks(y, stats_df['UniProt_ID'], fontsize=14)
    plt.xticks(fontsize=14)
    plt.xlim(0, 1)
    plt.xlabel(f'Spearman $\\rho$', fontsize=14)
    if group==1:
        plt.title(f'Inter-phenotype correlation', fontsize=16)
    else:
        plt.title(f'Inter-experimentor correlation', fontsize=16)
    plt.tight_layout()
    plt.savefig(os.path.join(png_dir,f'group-{group}_rho-barplot.png'),dpi=300,bbox_inches="tight")

    # === SCATTER FOR ALL PROTEINS ===
    y = np.arange(len(stats_df)) # plot all, not just the mean
    if group==1:
        hollow = ['KCNE1_HUMAN']
        filled = ['CP2C9_HUMAN','KCNJ2_MOUSE','OXDA_RHOTO','Q53Z42_HUMAN','S22A1_HUMAN','SPIKE_SARS2','VKOR1_HUMAN']
        plt.figure(figsize=(5.5, 5.0))
        plt.yticks(y, stats_df['UniProt_ID'], fontsize=14)
        legend_elements = [
            plt.scatter([], [], facecolors='none', edgecolors=color, s=50, linewidths=2.5, label='Binary fitness measurement'),
            plt.scatter([], [], color=color, s=50, label='Continuous fitness measurement')
        ]
        plt.legend(handles=legend_elements, fontsize=12,loc='upper center', bbox_to_anchor=(0.5, -0.15))
    elif group==2:
        plt.figure(figsize=(5.8, 4.2))
        plt.yticks(y, stats_df['UniProt_ID'], fontsize=14)
    for i, values in enumerate(stats_df['all_spearman']):
        if group == 1 and stats_df['UniProt_ID'].iloc[i] in hollow:
            plt.scatter(values, np.full(len(values), y[i]), 
                        facecolors='none', edgecolors=color, s=50, linewidths=2.5)
        else:
            plt.scatter(values, np.full(len(values), y[i]), color=color, s=50)
    plt.xticks(fontsize=14)
    plt.xlim(0, 1)
    plt.xlabel(f'Spearman $\\rho$', fontsize=14)
    if group==1:
        plt.title(f'Inter-phenotype correlation', fontsize=16)
    else:
        plt.title(f'Inter-experimentor correlation', fontsize=16)
    plt.grid(alpha=0.5)
    plt.tight_layout()
    plt.savefig(os.path.join(png_dir,f'group-{group}_rho-scatterplot.png'),dpi=300,bbox_inches="tight")


png_dir = '/home/pwoolley/work/proteingym/images/dataset_info/dataset_info_group1'
dataset_analysis_plots(g1_df,csv_dir,png_dir,group=1)
png_dir = '/home/pwoolley/work/proteingym/images/dataset_info/dataset_info_group2'
dataset_analysis_plots(g2_df,csv_dir,png_dir,group=2)

### Figure 3: Model performance for different datasets of the same measurement

Changes:
* Reorder so it's not alphabetical and instead by how well the different datasets correlate (from previous figure) (DONE)
* Make two versions, one rotated and one as is

In [ ]:
def make_model_group2_plots(g2_df,pkls,models,csv_dir):
    def tidy_data(g2_df):
        g2_df['group2_str'] = g2_df['group2_str'].str.split(',')
        g2_df = g2_df.explode('group2_str').reset_index(drop=True)
        all_scores = {v:[] for v in models.values()}
        all_scores['UniProt_ID'] = []
        pm = ProteinMutationModel(temperature=1.0)
        for i,row in g2_df.iterrows():
            proteinid = row['UniProt_ID']
            csv = os.path.join(csv_dir,row['group2_str'].split(':')[0])
            for pkl in pkls:
                try:
                    if pkl.split('/')[-1] not in models:
                        continue
                    model = models[pkl.split('/')[-1]]
                    pm.load_pickle(os.path.join(pkls_dir, pkl),protein_id=proteinid)
                    pm.compute_entropy()
                    pm.load_experiment(os.path.join(csv_dir, csv))
                    pm.build_dataset()
                    all_scores[model].append(pm.spearman())
                    passed = True
                except Exception as err:
                    passed = False
                    print(err)
                    break
            if passed:
                all_scores['UniProt_ID'].append(proteinid)
        all_scores = pd.DataFrame({k:v for k,v in all_scores.items() if len(v) > 0})
        all_scores = all_scores.sort_values('ESMC_600M',ascending=False).reset_index(drop=True)


    def plot(all_scores):
        plt.figure(figsize=(8, 6))
        offset = 0
        gap = 0
        score_cols = [c for c in all_scores.columns if c != "UniProt_ID"]
        ytick_positions = []
        ytick_labels = []
        colors = plt.cm.tab10(range(len(score_cols)))
        labeled = {col: False for col in score_cols}

        # Computing the col order
        score_cols = [c for c in all_scores.columns if c != "UniProt_ID"]
        group_means = (
            all_scores.groupby("UniProt_ID")[score_cols]
            .mean()
            .mean(axis=1)
            .sort_values()
        )
        col_order = group_means.index.tolist()

        for group in col_order:
            if group in all_scores["UniProt_ID"].values:
                gdf = all_scores[all_scores['UniProt_ID'] == group]
            else:
                continue
            gdf = gdf.sort_index()
            n = len(gdf)
            y = np.arange(n) + offset

            for i, col in enumerate(score_cols):
                color = colors[i]
                plt.plot(gdf[col], y, linestyle="-", alpha=0.6, color=color)
                plt.scatter(gdf[col], y, label=col if not labeled[col] else None, color=color)
                labeled[col] = True

            ytick_positions.append(y.mean())
            ytick_labels.append(group)
            boundary = offset + n - 0.5
            plt.axhline(boundary, linestyle="--", alpha=0.4)
            offset += n + gap

        plt.xlabel("Spearman $\\rho$",fontsize=14)
        plt.yticks(ytick_positions, ytick_labels,fontsize=14)
        plt.xticks(fontsize=14)
        handles, labels = plt.gca().get_legend_handles_labels()
        new_labels = [mapping.get(l, l) for l in labels]
        plt.legend(handles, new_labels, bbox_to_anchor=(1, 0.7), loc="upper left", fontsize=12)
        plt.tight_layout()
        plt.savefig('/home/pwoolley/work/proteingym/images/model_performance_groups/group2/group2_2_rotated.png',dpi=300,bbox_inches="tight")
        plt.show()

    all_scores = tidy_data(g2_df)
    plot(all_scores)
    return all_scores
groups = pd.read_csv('../DMS_substitutions_groups.tsv',sep='\t')
groups = groups[groups['UniProt_ID']!='SPG1_STRSG'] # manually dropping this row because its slow:
rep_score = 'esmc_600m.logits.x.combined.pkl'
all_scores = make_model_group2_plots(groups[groups['group2_member']==1],pkls,models,csv_dir,rep_score)

In [ ]:
def plot(all_scores):
    plt.figure(figsize=(8, 6))
    offset = 0
    gap = 0
    score_cols = [c for c in all_scores.columns if c != "UniProt_ID"]
    ytick_positions = []
    ytick_labels = []
    colors = plt.cm.tab10(range(len(score_cols)))
    labeled = {col: False for col in score_cols}

    # Computing the col order
    score_cols = [c for c in all_scores.columns if c != "UniProt_ID"]
    group_means = (
        all_scores.groupby("UniProt_ID")[score_cols]
        .mean()
        .mean(axis=1)
        .sort_values()
    )
    col_order = group_means.index.tolist()

    for group in col_order:
        if group in all_scores["UniProt_ID"].values:
            gdf = all_scores[all_scores['UniProt_ID'] == group]
        else:
            continue
        gdf = gdf.sort_index()
        n = len(gdf)
        y = np.arange(n) + offset

        for i, col in enumerate(score_cols):
            color = colors[i]
            plt.plot(gdf[col], y, linestyle="-", alpha=0.6, color=color)
            plt.scatter(gdf[col], y, label=col if not labeled[col] else None, color=color)
            labeled[col] = True

        ytick_positions.append(y.mean())
        ytick_labels.append(group)
        boundary = offset + n - 0.5
        plt.axhline(boundary, linestyle="--", alpha=0.4)
        offset += n + gap

    plt.xlabel("Spearman $\\rho$",fontsize=14)
    plt.yticks(ytick_positions, ytick_labels,fontsize=14)
    plt.xticks(fontsize=14)
    handles, labels = plt.gca().get_legend_handles_labels()
    new_labels = [mapping.get(l, l) for l in labels]
    plt.legend(handles, new_labels, bbox_to_anchor=(1, 0.7), loc="upper left", fontsize=12)
    plt.tight_layout()
    plt.savefig('/home/pwoolley/work/proteingym/images/model_performance_groups/group2/group2_2_rotated.png',dpi=300,bbox_inches="tight")
    plt.show()
plot(all_scores)

### Figure 3: Model performance for abundance vs activity

In [ ]:
def make_model_group1_plots(g1_df,pkls,models,csv_dir,png_dir):
    def linear_regression(x,y):
        m, b = np.polyfit(x, y, 1) # Pearson (linear fit)
        y_pred = m * x + b
        pearson_r2 = r2_score(y, y_pred)
        spearman_rho, _ = spearmanr(x, y) # Spearman
        spearman_r2 = spearman_rho**2
        return pearson_r2, spearman_rho, spearman_r2, y_pred
    
    def make_scatter_shape(df,title,png_dir):
        fig, ax = plt.subplots(figsize=(6, 5))
        sns.scatterplot(
            data=df,
            x="expression_rs",
            y="selection_rs",
            palette='rainbow',
            hue="variable_rs",
            style="model",
            s=100,
            ax=ax
        )
        # --- Extract shape-only handles from the auto legend ---
        handles, labels = ax.get_legend_handles_labels()
        # Seaborn puts hue entries first, then a "model" section header, then style entries
        # Find where the style section starts by locating the "model" header label
        style_start = labels.index("model")+1
        shape_handles = handles[style_start:]
        shape_labels  = labels[style_start:]

        # Replace variable names in shape labels using your map (shouldn't be needed
        # here since these are model names, but kept for consistency)
        shape_labels = [mapping.get(l, l) for l in shape_labels]

        ax.legend(shape_handles, shape_labels, bbox_to_anchor=(1.25, 0.8), loc="upper left",
              fontsize=12, title_fontsize=14)

        # --- Colorbar for variable_rs ---
        norm = mcolors.Normalize(vmin=df["variable_rs"].min(), vmax=df["variable_rs"].max())
        sm = cm.ScalarMappable(cmap="rainbow", norm=norm)
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=ax, pad=0.02, aspect=30)
        cbar.set_label("Inter-variable $\\rho$", fontsize=14)
        cbar.ax.tick_params(labelsize=11)

        # --- Rest unchanged ---
        xmin = min(df["expression_rs"].min(), df["selection_rs"].min())
        xmax = max(df["expression_rs"].max(), df["selection_rs"].max())
        ax.plot([xmin, xmax], [xmin, xmax], color="gray", linestyle="--", linewidth=1)
        ax.set_title(title, fontsize=16)
        ax.set_xlabel(f"Model, Abundance $\\rho$", fontsize=14)
        ax.set_ylabel(f"Model, Activity $\\rho$", fontsize=14)
        ax.tick_params(labelsize=14)

        plt.savefig(
            os.path.join(png_dir, "variable_rs_expression-activity-rs-scatter.png"),
            dpi=300, bbox_inches="tight"
        )
        plt.show()
    
    pm = ProteinMutationModel(temperature=1.0)
    all_scores = {'proteinid':[],'model':[],'expression_rs':[],'selection_rs':[],'selection_name':[]}
    all_protein_data = {'proteinid':[],'variable_rs':[]}
    for _, row in g1_df.iterrows():
        proteinid = row['UniProt_ID']
        # drop SARS2 which is a virus and not represented in all models' training data.
        if proteinid == 'SPIKE_SARS2':
            continue
        exps = row['group_str'].split(',')
        selections = []
        protein_data = []
        for exp in exps:
            selection = exp.split(':')[1]
            selections.append(selection)
            csv = os.path.join(csv_dir, exp.split(':')[0])
            exp_data = pd.read_csv(csv)[['mutant', 'DMS_score']]
            exp_data = exp_data.rename(columns={'DMS_score': selection})
            protein_data.append(exp_data)
            for pkl in pkls:
                try:
                    if pkl.split('/')[-1] not in models:
                        continue
                    model = models[pkl.split('/')[-1]]
                    pm.load_pickle(os.path.join(pkls_dir, pkl),protein_id=proteinid)
                    pm.compute_entropy()
                    pm.load_experiment(os.path.join(csv_dir, csv))
                    pm.build_dataset()
                    if selection == 'Abundance':
                        expression_rs = pm.spearman()
                        all_scores['expression_rs'].append(expression_rs)
                    else:
                        selection_rs = pm.spearman()
                        selection_name = selection
                        all_scores['selection_rs'].append(selection_rs)
                        all_scores['selection_name'].append(selection_name)
                        all_scores['proteinid'].append(proteinid)
                        all_scores['model'].append(model)
                except Exception as err:
                    print(err)
                    break
        protein_data = reduce(lambda left, right: pd.merge(left, right, on='mutant', how='inner'),protein_data)
        a,b = selections
        if b == "Abundance":
            y = protein_data[a]
            x = protein_data[b]
        else:
            x = protein_data[a]
            y = protein_data[b]
        try:
            _, spearman_rho, _, _ = linear_regression(x,y)
            all_protein_data['proteinid'].append(proteinid)
            all_protein_data['variable_rs'].append(spearman_rho)
        except:
            print(selections)
            break
    all_scores = pd.DataFrame(all_scores)
    all_protein_data = pd.DataFrame(all_protein_data)
    all_scores = pd.merge(all_scores,all_protein_data,how='left')
    make_scatter_shape(all_scores,'Model Spearman $\\rho$ for multiple phenotypes',png_dir)

    return all_scores

png_dir = '/home/pwoolley/work/proteingym/images/model_performance_groups/group1'
all_scores = make_model_group1_plots(g1_df,pkls,models,csv_dir,png_dir)

In [ ]:
def make_scatter_shape(df,title,png_dir):
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.scatterplot(
        data=df,
        x="expression_rs",
        y="selection_rs",
        palette='rainbow',
        hue="variable_rs",
        style="model",
        s=100,
        ax=ax
    )
    # --- Extract shape-only handles from the auto legend ---
    handles, labels = ax.get_legend_handles_labels()
    # Seaborn puts hue entries first, then a "model" section header, then style entries
    # Find where the style section starts by locating the "model" header label
    style_start = labels.index("model")+1
    shape_handles = handles[style_start:]
    shape_labels  = labels[style_start:]

    # Replace variable names in shape labels using your map (shouldn't be needed
    # here since these are model names, but kept for consistency)
    shape_labels = [mapping.get(l, l) for l in shape_labels]

    ax.legend(shape_handles, shape_labels, bbox_to_anchor=(1.25, 0.8), loc="upper left",
              fontsize=12, title_fontsize=14)

    # --- Colorbar for variable_rs ---
    norm = mcolors.Normalize(vmin=df["variable_rs"].min(), vmax=df["variable_rs"].max())
    sm = cm.ScalarMappable(cmap="rainbow", norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, pad=0.02, aspect=30)
    cbar.set_label("Inter-variable $\\rho$", fontsize=14)
    cbar.ax.tick_params(labelsize=11)

    # --- Rest unchanged ---
    xmin = min(df["expression_rs"].min(), df["selection_rs"].min())
    xmax = max(df["expression_rs"].max(), df["selection_rs"].max())
    ax.plot([xmin, xmax], [xmin, xmax], color="gray", linestyle="--", linewidth=1)
    # ax.set_title(title, fontsize=16)
    ax.set_xlabel(f"Model, Abundance $\\rho$", fontsize=14)
    ax.set_ylabel(f"Model, Activity $\\rho$", fontsize=14)
    ax.tick_params(labelsize=14)

    plt.savefig(
        os.path.join(png_dir, "variable_rs_expression-activity-rs-scatter.png"),
        dpi=300, bbox_inches="tight"
    )
    plt.show()
make_scatter_shape(all_scores,'Model Spearman $\\rho$ for multiple phenotypes',png_dir)

Fig 2 random: plotting KCNE1 outlier

In [ ]:
models

In [ ]:
csv = 'KCNE1_HUMAN_Muhammad_2023_function.csv'
# pkl = 'SaProt_650M_PDB.logits.x.combined.pkl'
# model = models[pkl]
for pkl,model in models.items():
    pm = ProteinMutationModel(temperature=1.0)
    colors = plt.cm.viridis(np.linspace(0, 1, 3))
    proteinid = '_'.join(csv.split('_')[:2])
    try:
        pm.load_pickle(os.path.join(pkls_dir, pkl),protein_id=proteinid)
    except:
        continue
    pm.compute_entropy()
    pm.load_experiment(os.path.join(csv_dir, csv))
    pm.build_dataset()
    fit_bubble = pm.data[pm.data['DMS_score_bin']==1]
    fit_min = fit_bubble['exp'].min()
    fittest_min = fit_bubble['exp'].median()
    fit_max = fit_bubble['exp'].max()
    bins = np.histogram_bin_edges(pm.data['exp'].values, bins=20)
    s = 2
    fig, ax = plt.subplots(figsize=(5.5, 5))
    ax.scatter(pm.data['log_odds'], pm.data['exp'], s=s, label='unfit', color=colors[0], alpha=0.4)
    ax.scatter(pm.data[pm.data['exp']>fit_min]['log_odds'], pm.data[pm.data['exp']>fit_min]['exp'], s=s, label='fit', color=colors[1], alpha=0.4)
    ax.scatter(pm.data[pm.data['exp']>fittest_min]['log_odds'], pm.data[pm.data['exp']>fittest_min]['exp'], s=s, label='fittest', color=colors[2], alpha=0.4)
    ax.set_title(f'{model} predictions for KCNE1 Activity', fontsize=16)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=13, markerscale=6)
    ax.axvline(0, c='k', linestyle='--')
    ax.set_xlabel('Mutation Log-Odds', fontsize=14)
    ax.set_ylabel('Abundance', fontsize=14)
    ax.tick_params(axis='both', labelsize=14)
    ax.xaxis.set_major_locator(ticker.MaxNLocator(nbins=5))
    ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=5))

    # plt.savefig(os.path.join(png_dir, "fitness_scatter.png"), dpi=300, bbox_inches="tight")
    plt.show()

### Figure S1: WT accuracy for each model
Works as a sanity check, but would be better as a table.

In [ ]:
pm = ProteinMutationModel(temperature=1.0)
csv_dir = '/home/pwoolley/work/proteingym/DMS_ProteinGym_substitutions'
csvs = [x for x in os.listdir(csv_dir)]
csv = csvs[0]
pkls_dir
for pkl in pkls:
    pm.load_pickle(pkl,protein_id='OBSCN_HUMAN')
    pm.compute_entropy()
    pm.load_experiment(os.path.join(csv_dir, csv))
    pm.build_dataset()
    plt.hist(pm.data[['index','wt_rank']].drop_duplicates()['wt_rank'],bins=np.arange(1,21,1))
    # plt.title(os.path.basename(pkl).split('.')[0])
    plt.xticks(np.arange(1,21,1))
    plt.show()

### Figure 4: How to interpret model probabilities for exploring mutational space
* Are $\rho$ values merely demonstrating avoiding poor mutations?
* Split up data below 0 and data above 0?

#### Fig 4A

In [ ]:
png_dir = '/home/pwoolley/work/proteingym/images/interpreting_correlation'
csvs = [x for x in os.listdir(csv_dir) if ('HUMAN' in x) and (('activity' in x) | ('abundance' in x))]
csv = csvs[0]
pkl = 'ESM3_sm_open_v0.both.logits.x.combined.pkl'
model = models[pkl]
pm = ProteinMutationModel(temperature=1.0)
colors = plt.cm.viridis(np.linspace(0, 1, 3))
proteinid = '_'.join(csv.split('_')[:2])
pm.load_pickle(os.path.join(pkls_dir, pkl),protein_id=proteinid)
pm.compute_entropy()
pm.load_experiment(os.path.join(csv_dir, csv))
pm.build_dataset()
fit_bubble = pm.data[pm.data['DMS_score_bin']==1]
fit_min = fit_bubble['exp'].min()
fittest_min = fit_bubble['exp'].median()
fit_max = fit_bubble['exp'].max()
bins = np.histogram_bin_edges(pm.data['exp'].values, bins=20)
s = 2
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.scatter(pm.data['log_odds'], pm.data['exp'], s=s, label='unfit', color=colors[0], alpha=0.4)
ax.scatter(pm.data[pm.data['exp']>fit_min]['log_odds'], pm.data[pm.data['exp']>fit_min]['exp'], s=s, label='fit', color=colors[1], alpha=0.4)
ax.scatter(pm.data[pm.data['exp']>fittest_min]['log_odds'], pm.data[pm.data['exp']>fittest_min]['exp'], s=s, label='fittest', color=colors[2], alpha=0.4)
# ax.set_title(f'ESM3 (hybrid) predictions for CYP2C9 Cytochrome', fontsize=16)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=13, markerscale=6)
ax.axvline(0, c='k', linestyle='--')
ax.set_xlabel('Mutation Log-Odds', fontsize=14)
ax.set_ylabel('Abundance', fontsize=14)
ax.tick_params(axis='both', labelsize=14)
ax.xaxis.set_major_locator(ticker.MaxNLocator(nbins=5))
ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=5))

plt.savefig(os.path.join(png_dir, "fitness_scatter.png"), dpi=300, bbox_inches="tight")
plt.show()

#### Fig 4B

In [ ]:
def make_fit_bubble_figures(pkls,csvs,png_dir):
    def make_density_plot(plt_data,png_dir):
        colors = plt.cm.viridis(np.linspace(0, 1, 3))
        fig, ax = plt.subplots(figsize=(6, 5))

        sns.kdeplot(plt_data['all_rs'], label='All Mutation $\\rho$', fill=True, alpha=0.3, color='k', ax=ax)
        sns.kdeplot(plt_data['unfit_rs'], label='Unfit Mutation $\\rho$', fill=True, alpha=0.7, color=colors[0], ax=ax)
        sns.kdeplot(plt_data['fit_rs'], label='Fit Mutation $\\rho$', fill=True, alpha=0.7, color=colors[1], ax=ax)
        sns.kdeplot(plt_data['fittest_rs'], label='Fittest Mutation $\\rho$', fill=True, alpha=0.7, color=colors[2], ax=ax)

        ax.axvline(plt_data['unfit_rs'].mean(), color=colors[0], linestyle='--', linewidth=2)
        ax.axvline(plt_data['fit_rs'].mean(), color=colors[1], linestyle='--', linewidth=2)
        ax.axvline(plt_data['fittest_rs'].mean(), color=colors[2], linestyle='--', linewidth=2)

        # ax.set_title('ESM3 (hybrid) Spearman $\\rho$ for mutation sets', fontsize=16)
        ax.tick_params(axis='both', labelsize=14)
        ax.set_xlabel('Spearman $\\rho$', fontsize=14)
        ax.set_ylabel('Probability Density', fontsize=14)

        ax.xaxis.set_major_locator(ticker.MaxNLocator(nbins=5))
        ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=5))

        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left',fontsize=13)
        plt.savefig(os.path.join(png_dir, f'all-spearmanr_fit-spearmanr_density.png'), dpi=300, bbox_inches="tight")
        plt.show()


    def data_pass(pkls, csvs):        
        model_set = [models[x] for x in pkls]
        pm = ProteinMutationModel(temperature=1.0)
        fitbubble_rs = {'all_rs':[],'unfit_rs':[],'fit_rs':[],'fittest_rs':[],'model':[]}
        entropybin_fitlikelihood = {'proteinid':[],'index':[],'entropy':[],'wt_picked':[],'fit':[],'fittest':[],'model':[],'log_odds':[]}

        fitbubble_ent_exp_wtbin = {'entropy':[],'exp':[],'wt_rank':[]}
        for csv in csvs:
            proteinid = '_'.join(csv.split('_')[:2])
            try:
                for pkl,model in zip(pkls,model_set):
                    model = models[pkl]
                    pm.load_pickle(os.path.join(pkls_dir, pkl),protein_id=proteinid)
                    pm.compute_entropy()
                    pm.load_experiment(os.path.join(csv_dir, csv))
                    pm.build_dataset()
                    fit_bubble = pm.data[pm.data['DMS_score_bin']==1]
                    all_rs = pm.spearman()
                    all_min = pm.data['exp'].min()
                    fit_min = fit_bubble['exp'].min()
                    fittest_min = fit_bubble['exp'].median()
                    fit_max = fit_bubble['exp'].max()
                    # for overlapping sets
                    # fit_rs = pm.spearman(subsetting={'exp':{'min_val':fit_min,'max_val':fit_max}})
                    # for non-overlapping sets:
                    unfit_rs = pm.spearman(subsetting={'exp':{'min_val':all_min,'max_val':fit_min}})
                    fit_rs = pm.spearman(subsetting={'exp':{'min_val':fit_min,'max_val':fittest_min}})
                    fittest_rs = pm.spearman(subsetting={'exp':{'min_val':fittest_min,'max_val':fit_max}})
                    fitbubble_rs['all_rs'].append(all_rs)
                    fitbubble_rs['unfit_rs'].append(unfit_rs)
                    fitbubble_rs['fit_rs'].append(fit_rs)
                    fitbubble_rs['fittest_rs'].append(fittest_rs)
                    fitbubble_rs['model'].append(model)
                    fitbubble_ent_exp_wtbin['entropy'].extend(fit_bubble['entropy'])
                    fitbubble_ent_exp_wtbin['exp'].extend(fit_bubble['exp'])
                    fitbubble_ent_exp_wtbin['wt_rank'].extend(fit_bubble['wt_rank'])

                    # position data
                    entropybin_fitlikelihood['proteinid'].extend([proteinid]*len(pm.data))
                    entropybin_fitlikelihood['index'].extend(pm.data['index'])
                    entropybin_fitlikelihood['entropy'].extend(pm.data['entropy'])
                    entropybin_fitlikelihood['wt_picked'].extend((pm.data['wt_rank']==1))
                    entropybin_fitlikelihood['fit'].extend((pm.data['DMS_score_bin']==1))
                    entropybin_fitlikelihood['fittest'].extend((pm.data['exp']>=fittest_min))
                    entropybin_fitlikelihood['model'].extend([model]*len(pm.data))
                    entropybin_fitlikelihood['log_odds'].extend(pm.data['log_odds'])


            except Exception as e:
                print(e)
                continue
        fitbubble_rs = pd.DataFrame(fitbubble_rs)
        fitbubble_ent_exp_wtbin = pd.DataFrame(fitbubble_ent_exp_wtbin)
        entropybin_fitlikelihood = pd.DataFrame(entropybin_fitlikelihood)

        test_df = entropybin_fitlikelihood.copy()
        entropybin_fitlikelihood = entropybin_fitlikelihood.drop(['log_odds'],axis=1)

        entropybin_fitlikelihood = entropybin_fitlikelihood.groupby(["proteinid", "index", "model"]).apply(lambda g: pd.Series({
                "entropy": g["entropy"].iloc[0],
                "wt_picked": g["wt_picked"].iloc[0],
                "fit_likelihood": g["fit"].mean(),
                "fittest_likelihood": g["fittest"].mean()
            })).reset_index()
        entropybin_fitlikelihood["entropy_bin"] = pd.cut(entropybin_fitlikelihood["entropy"], bins=10)
        entropybin_fitlikelihood['flexible'] = (entropybin_fitlikelihood['fit_likelihood']>0.3)
        return fitbubble_rs, fitbubble_ent_exp_wtbin, entropybin_fitlikelihood, test_df

    fitbubble_rs, fitbubble_ent_exp_wtbin, entropybin_fitlikelihood, test_df = data_pass(pkls, csvs)
    #make_density_plot(fitbubble_rs,png_dir)
    return fitbubble_rs, fitbubble_ent_exp_wtbin, entropybin_fitlikelihood, test_df
    
    
png_dir = '/home/pwoolley/work/proteingym/images/interpreting_correlation'
pkl_set = [k for k in models.keys() if 'ESM3_sm_open_v0.both' in k]
print(pkl_set)
csv_dir = '/home/pwoolley/work/proteingym/DMS_ProteinGym_substitutions'
csvs = [x for x in os.listdir(csv_dir)]
fitbubble_rs, fitbubble_ent_exp_wtbin, entropybin_fitlikelihood, test_df = make_fit_bubble_figures(pkl_set,csvs[:],png_dir)

Density plot (DONE)

In [ ]:
def make_density_plot(plt_data,png_dir):
    colors = plt.cm.viridis(np.linspace(0, 1, 3))
    fig, ax = plt.subplots(figsize=(6, 5))

    sns.kdeplot(plt_data['all_rs'], label='All Mutation $\\rho$', fill=True, alpha=0.3, color='k', ax=ax)
    sns.kdeplot(plt_data['unfit_rs'], label='Unfit Mutation $\\rho$', fill=True, alpha=0.7, color=colors[0], ax=ax)
    sns.kdeplot(plt_data['fit_rs'], label='Fit Mutation $\\rho$', fill=True, alpha=0.7, color=colors[1], ax=ax)
    sns.kdeplot(plt_data['fittest_rs'], label='Fittest Mutation $\\rho$', fill=True, alpha=0.7, color=colors[2], ax=ax)

    ax.axvline(plt_data['unfit_rs'].mean(), color=colors[0], linestyle='--', linewidth=2)
    ax.axvline(plt_data['fit_rs'].mean(), color=colors[1], linestyle='--', linewidth=2)
    ax.axvline(plt_data['fittest_rs'].mean(), color=colors[2], linestyle='--', linewidth=2)

    # ax.set_title('ESM3 (hybrid) Spearman $\\rho$ for mutation sets', fontsize=16)
    ax.tick_params(axis='both', labelsize=14)
    ax.set_xlabel('Spearman $\\rho$', fontsize=14)
    ax.set_ylabel('Probability Density', fontsize=14)

    ax.xaxis.set_major_locator(ticker.MaxNLocator(nbins=5))
    ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=5))

    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left',fontsize=13)
    plt.savefig(os.path.join(png_dir, f'all-spearmanr_fit-spearmanr_density.png'), dpi=300, bbox_inches="tight")
    plt.show()
make_density_plot(fitbubble_rs,'/home/pwoolley/work/proteingym/images/interpreting_correlation')

# _________________________________________

### Figure 5: New-to-nature functions, by protein
This does not use the same Protein class as before, it's all done by hand.

* Ozbek_2007 and Tawfik_2012 are the sequential ones

In [ ]:
Tawfik_2012_preds_name = ['OPD_BREDI.MODEL.logits.indices_254_306_274_233_172_269_272_80_111_204_130_271',
                          'OPD_BREDI_H254R.MODEL.logits.indices_233_306',
                          'OPD_BREDI_H254R_F306I.MODEL.logits.indices_274',
                          'OPD_BREDI_H254R_F306I_I274S.MODEL.logits.indices_233',
                          'OPD_BREDI_H254R_D233E_F306I_I274S.MODEL.logits.indices_172_269',
                          'OPD_BREDI_H254R_D233E_F306I_I274S_T172I.MODEL.logits.indices_269',
                          'OPD_BREDI_H254R_D233E_F306I_I274S_T172I_S269T_M138I_T199I.MODEL.logits.indices_272',
                          'OPD_BREDI_H254R_D233E_F306I_I274S_T172I_S269T_M138I_T199I_L272M.MODEL.logits.indices_80',
                          'OPD_BREDI_H254R_D233E_F306I_I274S_T172I_S269T_M138I_T199I_L272M_A80V.MODEL.logits.indices_111_204',
                          'OPD_BREDI_H254R_D233E_F306I_I274S_T172I_S269T_M138I_T199I_L272M_A80V_S111R.MODEL.logits.indices_204',
                          'OPD_BREDI_H254R_D233E_F306I_I274S_T172I_S269T_M138I_T199I_L272M_A80V_S111R_A204G.MODEL.logits.indices_130_271',
                          'OPD_BREDI_H254R_D233E_F306I_I274S_T172I_S269T_M138I_T199I_L272M_A80V_S111R_A204G_L130V.MODEL.logits.indices_271']
groups = {'Arnold_2012':{'csv':'/home/pwoolley/work/proteingym/fig4_data/CPXB_PRIM2_Arnold_2012.csv','preds_name':'CPXB_PRIM2.MODEL.logits.indices_182_264_329_401_438_439'},
          'Arnold_2014':{'csv':'/home/pwoolley/work/proteingym/fig4_data/CPXB_PRIM2_Arnold_2014.csv','preds_name':'CPXB_PRIM2.MODEL.logits.indices_182_264_329_401_438_439'},
          'Kaneko_2012':{'csv':'/home/pwoolley/work/proteingym/fig4_data/C1F2K5_ACIC5_Kaneko_2012.csv','preds_name':'C1F2K5_ACIC5.MODEL.logits.indices_45_292_334'},
          'Quax_2002':{'csv':'/home/pwoolley/work/proteingym/fig4_data/G7AC_PSEU7_Quax_2002.csv','preds_name':'G7AC_PSEU7.MODEL.logits.indices_178'},
          'full-Ozbek_2007':{'csv':'/home/pwoolley/work/proteingym/fig4_data/full-Q8IT70_HYDVU_Ozbek_2007_round-order.csv','preds_name':['full-Q8IT70_HYDVU.MODEL.logits.indices_482','full-Q8IT70_HYDVU_K482P.MODEL.logits.indices_472']},
          'truncated-Ozbek_2007':{'csv':'/home/pwoolley/work/proteingym/fig4_data/truncated-Q8IT70_HYDVU_Ozbek_2007_round-order.csv','preds_name':['truncated-Q8IT70_HYDVU.MODEL.logits.indices_21','truncated-Q8IT70_HYDVU_K21P.MODEL.logits.indices_11']},
          'Tawfik_2012':{'csv':'/home/pwoolley/work/proteingym/fig4_data/OPD_BREDI_Tawfik_2012_round-order.csv','preds_name':Tawfik_2012_preds_name}}

In [ ]:
def pull_probs(mutation, preds, preds_name):
    def softmax(x, axis=-1, temperature=1.0):
        if temperature <= 0:
            raise ValueError("Temperature must be positive.")
        x_scaled = x / temperature
        e_x = np.exp(x_scaled - np.max(x_scaled, axis=axis, keepdims=True))
        return e_x / np.sum(e_x, axis=axis, keepdims=True)
    
    def entropy(prob_dist):
        prob_dist = np.asarray(prob_dist)
        prob_dist = prob_dist[prob_dist > 0]
        return -np.sum(prob_dist * np.log2(prob_dist))

    def plot_probs(probs):
        plt.bar(AMINO_ACIDS,probs)
        # plt.xticks(labels=AMINO_ACIDS)
        plt.show()
        
    AMINO_ACIDS = ['A','R','N','D','C','Q','E','G','H','I','L','K','M','F','P','S','T','W','Y','V']
    wt_aa = mutation[0]
    mut_aa = mutation[-1]
    prot_index = mutation[1:-1]
    array_index = preds_name.split('.indices_')[-1].split('_')
    array_index = array_index.index(prot_index)
    pred = preds[array_index]
    probs = softmax(pred)
    ent = entropy(probs)
    wt_index = AMINO_ACIDS.index(wt_aa)
    mut_index = AMINO_ACIDS.index(mut_aa)
    wt_prob = probs[wt_index]
    mut_prob = probs[mut_index]
    # plot_probs(probs)
    return wt_prob,mut_prob,np.log(mut_prob/wt_prob),ent

In [ ]:
model = 'SaProt_650M_PDB'
pkl = f'/home/pwoolley/work/proteingym/outputs/logits/denovo/{model}.logits.x.denovo.pkl'

# model = 'protein_mpnn.proteinmpnn_v_48_020'
# pkl = f'/home/pwoolley/work/proteingym/outputs/logits/denovo/{model}.logits.x.denovo.use_sequence0.pkl'

#### Ozbek_2007

In [ ]:
group = 'full-Ozbek_2007'
csv = groups[group]['csv']
png_dir = '/home/pwoolley/work/proteingym/images/denovo'
preds_names = [x.replace('MODEL',model) for x in groups[group]['preds_name']]
with open(pkl,'rb') as inf:
    all_preds = pickle.load(inf)
data = pd.read_csv(csv)
bar1 = []
bar2 = []
muts_p = []
muts = []
round = []

muts_p_wtbackground = []
#wt_preds = all_preds['truncated-Q8IT70_HYDVU.ESM3_sm_open_v0.both.logits.indices_11_21']
wt_preds = all_preds[f'full-Q8IT70_HYDVU.{model}.logits.indices_472_482']

for i,row in data.iterrows():
    if row['name']=='WT':
        bar1.append(row['pose1'])
        bar2.append(row['pose2'])
        muts_p.append(np.nan)
        muts_p_wtbackground.append(np.nan)
        muts.append('WT')
        round.append(row['round'])
        continue
    bar1.append(row['pose1'])
    bar2.append(row['pose2'])
    position = row['mutant'][1:-1]
    pred_name = row['pred'].replace('MODEL', model)
    preds = all_preds[pred_name]
    _,mut_p,_,_ = pull_probs(row['mutant'],preds,pred_name)
    _,mut_p_wtbackground,_,_ = pull_probs(row['mutant'],wt_preds,pred_name)
    muts_p.append(mut_p)
    muts_p_wtbackground.append(mut_p_wtbackground)
    muts.append(row['mutant'])
    round.append(row['round'])
round = np.array(round)
fig, ax1 = plt.subplots(figsize=(5.5, 5))
ax1.plot(round, muts_p, color="#000000", marker='o',markersize=7, linewidth=3, label='Probability', alpha=0.8)
ax1.plot(round, muts_p_wtbackground, color="#000000", marker='o',markersize=7, linewidth=3, linestyle='--', label='Probability', alpha=0.8)
ax1.set_ylabel('Probability', fontsize=14)
ax1.set_ylim(0, 1)
ax2 = ax1.twinx()
width = 0.33
colors = ['#1f77b4', '#ff7f0e']  # Distinct colors for each metric
for i, (metric, label, color) in enumerate(zip([bar1,bar2], ['Pose 1','Pose 2'], colors)):
    offset = (i - 0.5) * width
    values = metric
    ax2.bar(round + offset, values, width, label=label, color=color, alpha=0.8)
# ax1.set_xlabel('Mutation Round and Mutant', fontsize=14)
ax1.set_xlabel('Mutation Round', fontsize=14)
ax2.set_ylabel('Pose percentage', fontsize=14)
ax2.set_xticks(round)
# xtick_labels = [f"{r}\n{m}" for m, r in zip(muts, round)]
xtick_labels = [f"{r}" for r in round]
ax2.set_xticklabels(xtick_labels)
ax2.set_ylim(0,100)
ax1.set_zorder(2)
ax2.set_zorder(1)
ax1.tick_params(axis='y', labelsize=14)
ax1.tick_params(axis='x', labelsize=14)
ax2.tick_params(axis='y', labelsize=14)
ax1.patch.set_visible(False)
ax2.legend(bbox_to_anchor=(0.85, -0.20), loc='upper center',fontsize=12)
from matplotlib.lines import Line2D
linestyle_legend_elements = [
    Line2D([0], [0], color='black', linewidth=3, linestyle='-',  label='Directed Evolution Background'),
    Line2D([0], [0], color='black', linewidth=3, linestyle='--', label='WT Background'),
]
linestyle_legend = ax1.legend(handles=linestyle_legend_elements, bbox_to_anchor=(0.3, -0.20), loc='upper center', fontsize=11)
ax1.add_artist(linestyle_legend)
plt.title('Hydra minicollagen pose flip',fontsize=16)
plt.tight_layout()
plt.savefig(os.path.join(png_dir,f'Ozbek-2007_{model}.png'),dpi=300,bbox_inches="tight")
plt.show()

#### Tawfik_2012

In [ ]:
group = 'Tawfik_2012'
csv = groups[group]['csv']
png_dir = '/home/pwoolley/work/proteingym/images/denovo'
preds_names = [x.replace('MODEL',model) for x in groups[group]['preds_name']]
with open(pkl,'rb') as inf:
    all_preds = pickle.load(inf)
data = pd.read_csv(csv)
bar1 = []
bar2 = []
muts_p = []
muts = []
round = []

muts_p_wtbackground = []
wt_preds = all_preds[f'OPD_BREDI.{model}.logits.indices_80_111_130_172_204_233_254_269_271_272_274_306']

for i,row in data.iterrows():
    if row['name']=='WT':
        bar1.append(row['kcat-Km_2NH'])
        bar2.append(row['kcat-Km_Paroxon'])
        muts_p.append(np.nan)
        muts_p_wtbackground.append(np.nan)
        muts.append('WT')
        round.append(row['round'])
        continue
    bar1.append(row['kcat-Km_2NH'])
    bar2.append(row['kcat-Km_Paroxon'])
    position = row['mutant'][1:-1]
    pred_name = row['pred'].replace('MODEL', model)
    preds = all_preds[pred_name]
    _,mut_p,_,_ = pull_probs(row['mutant'],preds,pred_name)
    _,mut_p_wtbackground,_,_ = pull_probs(row['mutant'],wt_preds,pred_name)
    muts_p.append(mut_p)
    muts_p_wtbackground.append(mut_p_wtbackground)
    muts.append(row['mutant'])
    round.append(row['round'])
round = np.array(round)
fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(round, muts_p, color="#000000", marker='o',markersize=7, linewidth=3, label='Probability', alpha=0.8)
ax1.plot(round, muts_p_wtbackground, color="#000000", marker='o',markersize=7, linewidth=3, linestyle='--', label='Probability', alpha=0.8)
ax1.set_ylabel('Probability', fontsize=14)
ax1.set_ylim(0, 1)
ax2 = ax1.twinx()
width = 0.33
colors = ['#1f77b4', '#ff7f0e']  # Distinct colors for each metric
for i, (metric, label, color) in enumerate(zip([bar1,bar2], ['2NH','Paroxon'], colors)):
    offset = (i - 0.5) * width
    values = metric
    ax2.bar(round + offset, values, width, label=label, color=color, alpha=0.8)
# ax1.set_xlabel('Mutation Round and Mutant', fontsize=14)
ax1.set_xlabel('Mutation Round', fontsize=14)
ax2.set_ylabel('$k_{\t{cat}}/K_m$', fontsize=14)
ax2.set_xticks(round)
# xtick_labels = [f"{r}\n{m}" for m, r in zip(muts, round)]
xtick_labels = [f"{r}" for r in round]
ax2.set_xticklabels(xtick_labels)
ax2.axvline(x=7, linestyle="--", color="gray", alpha=0.5)
ax2.axvline(x=8, linestyle="--", color="gray", alpha=0.5)
ax2.set_yscale('log')
ax2.set_ylim(100)
ax1.set_zorder(2)
ax2.set_zorder(1)
ax1.tick_params(axis='y', labelsize=14)
ax1.tick_params(axis='x', labelsize=14)
ax2.tick_params(axis='y', labelsize=14)
ax1.patch.set_visible(False)
ax2.legend(bbox_to_anchor=(0.64, -0.20), loc='upper center',fontsize=12)
from matplotlib.lines import Line2D
linestyle_legend_elements = [
    Line2D([0], [0], color='black', linewidth=3, linestyle='-',  label='Directed Evolution Background'),
    Line2D([0], [0], color='black', linewidth=3, linestyle='--', label='WT Background'),
]
linestyle_legend = ax1.legend(handles=linestyle_legend_elements, bbox_to_anchor=(0.34, -0.20), loc='upper center', fontsize=11)
ax1.add_artist(linestyle_legend)
plt.title('Phosphotriesterase Hydrolysis of Paroxon Substrate',fontsize=16)
plt.tight_layout()
plt.savefig(os.path.join(png_dir,f'Tawfik_2012_{model}.png'),dpi=300,bbox_inches="tight")
plt.show()

#### Arnold 2012 (DONE)

In [ ]:
group = 'Arnold_2012'
i = 0 # 0 is esmc, 1 is mpnn
csv = groups[group]['csv']
png_dir = '/home/pwoolley/work/proteingym/images/denovo'
preds_name = groups[group]['preds_name'].replace('MODEL',model)
with open(pkl,'rb') as inf:
    all_preds = pickle.load(inf)
preds = all_preds[preds_name]
data = pd.read_csv(csv)

plt_data = data.sort_values('trans_pct')
plt_data['position'] = plt_data['mutant'].apply(lambda x: x[1:-1])
WT_ROW = plt_data.loc[plt_data["mutant"] == "WT"].iloc[0]
plt_data = plt_data[plt_data['mutant']!='WT']
positions = sorted(plt_data["position"].dropna().unique())

x_bar = []
x_line = []
trans_vals = []
probs = []
colors = []
labels = []
separators = []
group_centers = []

x_ctr = 0
GROUP_GAP = 1.0
for pos in positions:
    start_x = x_ctr
    sub = plt_data[plt_data["position"] == pos].copy()
    wt_res = sub.iloc[0]['mutant'][0]
    # add synthetic WT-at-position
    wt_pos = {
        "mutant": f"{wt_res}{pos}{wt_res}",
        "trans_pct": WT_ROW["trans_pct"],
        "cis_pct": WT_ROW["cis_pct"],
        "position": pos
    }
    sub = pd.concat([pd.DataFrame([wt_pos]), sub], ignore_index=True)
    # sort by trans_pct
    sub = sub.sort_values("trans_pct")
    for _, row in sub.iterrows():
        x_bar.append(x_ctr)
        x_line.append(x_ctr)
        trans_vals.append(row["trans_pct"])
        _,mut_p,_,_ = pull_probs(row['mutant'],preds,preds_name)
        probs.append(mut_p)
        colors.append("tab:green" if (row["mutant"][0]==row["mutant"][-1]) else "tab:blue")
        labels.append(row["mutant"][-1])
        x_ctr += 1
    end_x = x_ctr - 1
    group_centers.append((start_x + end_x) / 2)
    x_line.append(np.nan)
    probs.append(np.nan)
    x_ctr += GROUP_GAP
    separators.append(x_ctr - 0.5)
    x_ctr += GROUP_GAP

# # --- plot ---
# fig, ax1 = plt.subplots(figsize=(13, 5))
# ax1.plot(x_line, probs, color="#000000", marker='o',markersize=7, linewidth=3, label='Probability', alpha=0.8)
# ax1.set_ylabel('Probability', fontsize=12, fontweight='bold')
# ax1.set_ylim(0, 1)
# for tick in ax1.get_yticklabels():
#     tick.set_fontweight('bold')
# # bars
# ax2 = ax1.twinx()
# ax2.bar(x_bar, trans_vals, color=colors)
# ax2.set_ylabel("Trans Isomer Percent")
# # dashed separators between positions
# for s in separators[:-1]:
#     ax2.axvline(s, linestyle="--", color="gray", alpha=0.5)
# ax2.set_xticks(x_bar)
# ax_pos = ax2.twiny()
# ax_pos.set_xlim(ax2.get_xlim())
# ax_pos.set_xticks(group_centers)
# ax_pos.set_xticklabels(positions, fontweight='bold')
# ax_pos.xaxis.set_ticks_position('bottom')
# ax_pos.xaxis.set_label_position('bottom')
# ax_pos.spines['bottom'].set_position(('outward', 30))
# ax_pos.spines['top'].set_visible(False)
# ax_pos.spines['right'].set_visible(False)
# ax_pos.spines['left'].set_visible(False)
# ax_pos.set_xlabel("Position", fontsize=12, fontweight='bold')
# ax2.set_xticklabels(labels)
# ax1.set_zorder(2)
# ax2.set_zorder(1)
# ax1.patch.set_visible(False)
# plt.title('Cytochrome P450 Isomer Engineering')
# plt.tight_layout()
# plt.savefig(os.path.join(png_dir,f'Arnold_2012_bar.png'),dpi=300,bbox_inches="tight")
# plt.show()

plt.figure(figsize=(4,4))
legend_elements = [
    Line2D([0], [0], marker='o', color='w',
           markerfacecolor="tab:green", markersize=9, label='WT'),
    Line2D([0], [0], marker='o', color='w',
           markerfacecolor="tab:blue", markersize=9, label='Mutant'),
]
plt.legend(handles=legend_elements,fontsize=14,bbox_to_anchor=(1.0, 0.7), loc='upper left')
plt.title('Cytochrome P450 Trans Isomer',fontsize=15)
plt.scatter([x for x in probs if not np.isnan(x)], trans_vals,c=colors,s=125)
plt.xlabel('Probability',fontsize=14)
plt.ylabel('Trans Isomer Percentage', fontsize=14)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.savefig(os.path.join(png_dir,f'Arnold_2012_scatter_{model}.png'),dpi=300,bbox_inches="tight")
plt.show()

#### Arnold 2014 (DONE)

In [ ]:
group = 'Arnold_2014'
i = 0 # 0 is esmc, 1 is mpnn
csv = groups[group]['csv']
png_dir = '/home/pwoolley/work/proteingym/images/denovo'
preds_name = groups[group]['preds_name'].replace('MODEL',model)
with open(pkl,'rb') as inf:
    all_preds = pickle.load(inf)
preds = all_preds[preds_name]
data = pd.read_csv(csv)

plt_data = data.sort_values('TTN')
probs = []
for _, row in plt_data.iterrows():
    if row["mutant"] == 'WT':
        _,y,_,_ = pull_probs("C401C",preds, preds_name)
        probs.append(y)
    else:
        _,y,_,_ = pull_probs(row['mutant'],preds, preds_name)
        probs.append(y)
# fig, ax1 = plt.subplots(figsize=(8, 6))
# line = ax1.plot(plt_data['mutant'], probs, color="#000000", marker='o',markersize=7, linewidth=3, label='Probability', alpha=0.8)
# ax1.set_ylabel('Probability', fontsize=12, fontweight='bold')
# ax1.set_ylim(0, 1)
# for tick in ax1.get_yticklabels():
#     tick.set_fontweight('bold')
# # Second scatter plot (right y-axis)
# ax2 = ax1.twinx()
colors = ['tab:green' if m == 'WT' else 'tab:blue' for m in plt_data['mutant']]
# ax2.bar(plt_data['mutant'], plt_data['TTN'], linewidth=3, color=colors)
# ax2.set_xlabel('Mutant', fontsize=12, fontweight='bold')
# ax2.set_ylabel('TTN (product/min)', fontsize=12)
# #ax2.set_xticks(x)
# ax2.set_xticklabels(plt_data['mutant'], fontsize=11)
# ax2.set_yscale('log')
# ax1.set_zorder(2)
# ax2.set_zorder(1)
# ax1.patch.set_visible(False)
# plt.title('Cytochrome N,N-diethyl-2-phenylacrylamide Cycloproponation Engineering')
# plt.tight_layout()
# plt.savefig(os.path.join(png_dir,f'Arnold_2014.png'),dpi=300,bbox_inches="tight")
# plt.show()


plt.figure(figsize=(4,4))
legend_elements = [
    Line2D([0], [0], marker='o', color='w',
           markerfacecolor="tab:green", markersize=9, label='WT'),
    Line2D([0], [0], marker='o', color='w',
           markerfacecolor="tab:blue", markersize=9, label='Mutant'),
]
plt.legend(handles=legend_elements,fontsize=14,bbox_to_anchor=(1.0, 0.7), loc='upper left')
plt.title('Cytochrome P450 Trans Isomer',fontsize=15)
plt.scatter([x for x in probs if not np.isnan(x)], plt_data['TTN'],c=colors,s=125)
plt.xlabel('Probability',fontsize=14)
plt.ylabel('Trans Turnover Number', fontsize=14)
plt.xticks(fontsize=14)
plt.yticks([1000,3000,5000,7000],fontsize=14)
plt.savefig(os.path.join(png_dir,f'Arnold_2014_scatter_{model}.png'),dpi=300,bbox_inches="tight")
plt.show()

#### Kaneko_2012 (DONE)

In [ ]:
group = 'Kaneko_2012'
csv = groups[group]['csv']
png_dir = '/home/pwoolley/work/proteingym/images/denovo'
preds_name = groups[group]['preds_name'].replace('MODEL',model)
with open(pkl,'rb') as inf:
    all_preds = pickle.load(inf)
preds = all_preds[preds_name]
data = pd.read_csv(csv)

plt_data = data.iloc[:2,:]
wt_p,mut_p,_,_ = pull_probs('Y334F',preds,preds_name)
width = 0.25
metrics = ['PNP-β-GlcA_specificity-shift', 'PNP-β-Glc_specificity-shift', 'PNP-β-Xyl_specificity-shift']
metric_labels = ['PNP-β-GlcA', 'PNP-β-Glc', 'PNP-β-Xyl']
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']  # Distinct colors for each metric
x = np.arange(len(plt_data))
fig, ax1 = plt.subplots(figsize=(5, 5))  # wider figure
line = ax1.plot(x, [wt_p, mut_p], color="#000000", marker='o', markersize=7, linewidth=3, label='Probability', alpha=0.8)
ax1.set_ylabel('Probability', fontsize=14)
ax1.set_ylim(0, 1)
ax1.tick_params(axis='y', labelsize=14)  # y-tick fontsize
ax2 = ax1.twinx()
for i, (metric, label, color) in enumerate(zip(metrics, metric_labels, colors)):
    offset = (i - 1) * width
    values = plt_data[metric].values
    ax2.bar(x + offset, values, width, label=label, color=color, alpha=0.8)
ax2.set_xlabel('Mutant', fontsize=14)
ax2.set_ylabel('Specificity Shift', fontsize=14)
ax2.set_xticks(x)
ax1.set_xticklabels(plt_data['mutant'], fontsize=14)  # x-tick fontsize
ax2.tick_params(axis='y', labelsize=14)  # right y-axis tick fontsize
ax2.set_yscale('log')
ax2.set_ylim(0.1)
ax1.set_zorder(2)
ax2.set_zorder(1)
ax1.patch.set_visible(False)
ax2.legend(loc='upper center', bbox_to_anchor=(0.5, -0.1), bbox_transform=ax1.transAxes, fontsize=12)
plt.title('β-Glucuronidase Promiscuity', fontsize=16)
plt.tight_layout()
plt.savefig(os.path.join(png_dir, f'Kaneko_2012_{model}.png'), dpi=300, bbox_inches="tight")
plt.show()

#### Quax_2002 Omitting

In [ ]:
group = 'Quax_2002'
i = 0 # 0 is esmc, 1 is mpnn
csv = groups[group]['csv']
png_dir = '/home/pwoolley/work/proteingym/images/denovo'
model = 'ESM3_sm_open_v0.both'
pkl = '/home/pwoolley/work/proteingym/outputs/logits/denovo/ESM3_sm_open_v0.both.logits.x.denovo.pkl'
preds_name = groups[group]['preds_name'].replace('MODEL',model)
with open(pkl,'rb') as inf:
    all_preds = pickle.load(inf)
preds = all_preds[preds_name]
data = pd.read_csv(csv)

plt_data = data.sort_values('Adipyl-7-ADCA_specificity-shift')
probs = []
for _, row in plt_data.iterrows():
    if row["mutant"] == 'WT':
        _,y,_,_ = pull_probs("Y178Y",preds, preds_name)
        probs.append(y)
    else:
        _,y,_,_ = pull_probs(row['mutant'],preds, preds_name)
        probs.append(y)

# fig, ax1 = plt.subplots(figsize=(6, 5))
# # First scatter plot (left y-axis)
# scatter1 = ax1.plot(plt_data['mutant'],probs, color="#000000", marker='o',markersize=7, linewidth=3, label='Adipyl-7-ADCA specificity-shift')
# ax1.set_ylabel('ESMC-600m Probability', color="#000000")
# ax1.set_ylabel('Probability', fontsize=12, fontweight='bold')
# ax1.set_ylim(0, 1)
# for tick in ax1.get_yticklabels():
#     tick.set_fontweight('bold')
# # Second scatter plot (right y-axis)
# ax2 = ax1.twinx()
colors = ['tab:green' if m == 'WT' else 'tab:blue' for m in plt_data['mutant']]
# ax2.bar(plt_data['mutant'], plt_data['Adipyl-7-ADCA_specificity-shift'], linewidth=3, color=colors)
# ax2.set_ylabel('Specificity Shift')
# ax2.tick_params(axis='y')
# ax1.set_zorder(2)
# ax2.set_zorder(1)
# ax1.patch.set_visible(False)
# plt.title('Glutaryl Acylase Adipyl-7-ADCA Specificity Engineering')
# fig.tight_layout()
# lines1, labels1 = ax1.get_legend_handles_labels()
# lines2, labels2 = ax2.get_legend_handles_labels()
# plt.savefig(os.path.join(png_dir,f'Quax_2002.png'),dpi=300,bbox_inches="tight")
# plt.show()



plt.figure(figsize=(4,4))
legend_elements = [
    Line2D([0], [0], marker='o', color='w',
           markerfacecolor="tab:green", markersize=9, label='WT'),
    Line2D([0], [0], marker='o', color='w',
           markerfacecolor="tab:blue", markersize=9, label='Mutant'),
]
plt.legend(handles=legend_elements,fontsize=14,bbox_to_anchor=(1.0, 0.7), loc='upper left')
plt.title('Glutaryl Acylase Adipyl-7-ADCA Specificity',fontsize=15)
plt.scatter([x for x in probs if not np.isnan(x)], plt_data['Adipyl-7-ADCA_specificity-shift'],c=colors,s=125)
plt.xlabel('Probability',fontsize=14)
plt.ylabel('Specificity Shift', fontsize=14)
plt.xticks([0,0.2,0.4],fontsize=14)
plt.yticks([1,3,5,7],fontsize=14)
plt.show()


# ______________________________

### JSDivergence heatmap, supplement

In [ ]:
def make_similarity_figures(pkls,models):
    def softmax(x, axis=-1, temperature=1.0):
        if temperature <= 0:
            raise ValueError("Temperature must be positive.")
        x_scaled = x / temperature
        e_x = np.exp(x_scaled - np.max(x_scaled, axis=axis, keepdims=True))
        return e_x / np.sum(e_x, axis=axis, keepdims=True)
    
    def parse_key(key):
        """
        Parse keys like:
        truncated-Q8IT70_HYDVU.<model>.logits.indices_21
        """
        protein = key.split('.')[0]
        m = re.search(r'indices_(.+)$', key)
        if m is None:
            raise ValueError(f"Could not parse indices from key: {key}")
        indices = tuple(map(int, m.group(1).split('_')))
        return protein, indices

    def mean_scores_between_pickles(pkl_a, pkl_b, eps=1e-12):
        """
        Compute mean Jensen–Shannon divergence  and SpearmanR across all
        aligned entries in two multi-entry pickle files.
        """
        with open(pkl_a, "rb") as f:
            data_a = pickle.load(f)
        with open(pkl_b, "rb") as f:
            data_b = pickle.load(f)
        map_a = {} # Build lookup: (protein, indices) -> logits
        for k, v in data_a.items():
            map_a[parse_key(k)] = v
        map_b = {}
        for k, v in data_b.items():
            map_b[parse_key(k)] = v
        common_keys = map_a.keys() & map_b.keys() # Find common entries
        if not common_keys:
            raise ValueError("No overlapping (protein, indices) entries")
        jsd_vals = []
        spearmanr_vals = []
        for key in common_keys:
            logits_a = map_a[key]
            logits_b = map_b[key]
            if logits_a.shape != logits_b.shape:
                print(f"Shape mismatch:",logits_a.shape,logits_b.shape)
                print(key)
                continue
            probs_a = softmax(logits_a, axis=1).astype(np.float64) # logits -> probabilities
            probs_b = softmax(logits_b, axis=1).astype(np.float64)
            probs_a = np.clip(probs_a, eps, 1.0)
            probs_b = np.clip(probs_b, eps, 1.0)
            for p, q in zip(probs_a, probs_b): # JSD per position
                jsd_vals.append(jensenshannon(p, q, base=2.0) ** 2)
                rho, _ = spearmanr(p, q)
                if not np.isnan(rho):
                    spearmanr_vals.append(rho)
        return float(np.mean(jsd_vals)), float(np.mean(spearmanr_vals))
    
    def make_scores_dfs(pkls,models):
        pkls = [x for x in pkls if os.path.basename(x) in models]
        labels = [models[os.path.basename(p)] for p in pkls]
        jsd_df = pd.DataFrame(data=np.nan,index=labels,columns=labels,dtype=float)
        spearmanr_df = pd.DataFrame(data=np.nan,index=labels,columns=labels,dtype=float)
        # Diagonal = 0 by definition
        for lbl in labels:
            jsd_df.loc[lbl, lbl] = 0.0
            spearmanr_df.loc[lbl, lbl] = 1.0
        # Compute only upper triangle
        for (i, pkl_a), (j, pkl_b) in combinations(enumerate(pkls), 2):
            label_a = labels[i]
            label_b = labels[j]
            jsd,spearmanr = mean_scores_between_pickles(pkl_a, pkl_b)
            jsd_df.loc[label_a, label_b] = jsd
            jsd_df.loc[label_b, label_a] = jsd  # symmetry
            spearmanr_df.loc[label_a, label_b] = spearmanr
            spearmanr_df.loc[label_b, label_a] = spearmanr  # symmetry
        return jsd_df,spearmanr_df
    
    def plot_scores_heatmaps(score_df, figsize=(8, 6), cmap="viridis"):
        row_linkage = linkage(score_df.values, method='average')
        col_linkage = linkage(score_df.values.T, method='average')        
        row_order = leaves_list(row_linkage)
        col_order = leaves_list(col_linkage)        
        clustered_df = score_df.iloc[row_order, col_order]        
        fig, ax = plt.subplots(figsize=figsize)
        im = ax.imshow(clustered_df.values, cmap=cmap)        
        ax.set_xticks(np.arange(clustered_df.shape[1]))
        ax.set_yticks(np.arange(clustered_df.shape[0]))
        ax.set_xticklabels(clustered_df.columns, rotation=90)
        ax.set_yticklabels(clustered_df.index)
        cbar = plt.colorbar(im, ax=ax)
        cbar.set_label("Mean Jensen–Shannon Divergence (bits)")
        plt.tight_layout()
        plt.show()

    jsd_df,spearmanr_df = make_scores_dfs(pkls,models)
    plot_scores_heatmaps(jsd_df)
    plot_scores_heatmaps(spearmanr_df)
    return jsd_df,spearmanr_df

jsd_df,spearmanr_df = make_similarity_figures(pkls,models)

In [ ]:
def plot_scores_heatmaps(score_df,figsize=(8, 6),cmap="Blues",cbar_label="Mean Score",order=None):
    # If no order provided, compute one (shared for rows & cols)
    if order is None:
        linkage_mat = linkage(score_df.values, method="average")
        order = leaves_list(linkage_mat)
    clustered_df = score_df.iloc[order, order]
    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(clustered_df.values, cmap=cmap)
    ax.set_xticks(np.arange(clustered_df.shape[1]))
    ax.set_yticks(np.arange(clustered_df.shape[0]))
    ax.set_xticklabels(clustered_df.columns, rotation=90)
    ax.set_yticklabels(clustered_df.index)
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label(cbar_label)
    plt.tight_layout()
    plt.show()
    return order

order = plot_scores_heatmaps(jsd_df,cmap='Blues',cbar_label='Mean Jensen–Shannon Divergence (bits)')
plot_scores_heatmaps(spearmanr_df,cmap="Blues_r",cbar_label='Mean SpearmanR',order=order)

# ________________________
### SCRATCH

### Lowest performing proteins from ProteinGym

In [ ]:
pg_data = pd.read_csv('../stats.tsv',sep='\t')
pg_models = list(pg_data.columns[2:98])
pg_data['row_avg'] = pg_data[pg_models].mean(axis=1)

In [ ]:
pg_data.sort_values('row_avg').head(20)

### Plots

Scatterplot of log(odds) x experimental value. Good for showing that extremely high functioning mutations are not guessed.

In [ ]:
pkl = pkls[2]
model = models[pkl.split('/')[-1]]
exp = tidy_exp_probs(pkl,proteinid,csv)
exp = exp.rename(columns={'DMS_score':'exp','prob':'mut_prob'})
e_low = ProteinModelEvaluator(exp)
pkl = pkls[7]
model = models[pkl.split('/')[-1]]
exp = tidy_exp_probs(pkl,proteinid,csv)
exp = exp.rename(columns={'DMS_score':'exp','prob':'mut_prob'})
e_high = ProteinModelEvaluator(exp)

plt.scatter(e_high.df['log_odds'],e_high.df['exp'],s=1)
plt.title('ISMC_600M')
plt.xlim(-15,15)
plt.xlabel('log(P(mut_aa)/P(wt_aa))')
plt.ylabel('Experimental value')
plt.show()